# 🪕 KORA-lite — Notebook 4 : la démo

On assemble tout dans une **application web** :

```
Message écrit (FR / fon)   ──────────────────────────────────┐
Capture d'écran  → lecture du texte (OCR Tesseract) ──────────┤
Vocal en fon     → oreille fon (mms-300m-fongbe) → texte ────┼→ cerveau KORA → verdict + raisons + conseil + mascotte
Vocal en français → Whisper (OpenAI, licence MIT) → texte ───┘
```

La mascotte KORA réagit au verdict : **contente** si le message est sûr, **surprise** s'il est suspect, **fâchée** si c'est une arnaque.

Gradio crée un **lien public** (`https://….gradio.live`) valable 72 h, à ouvrir sur un téléphone pour la démo devant le jury.

⚙️ GPU T4 conseillé. ⏱ ~3 min de chargement.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip -q install "gradio>=6,<7" pytesseract
    !apt-get -qq install -y tesseract-ocr tesseract-ocr-fra > /dev/null

In [ ]:
# ▶ Connexion à Google Drive et chemins du projet
# En Colab : le dossier KORA doit être dans "Mon Drive" (MyDrive/KORA).
# En local : le notebook utilise le dossier parent.
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/KORA"
else:
    BASE = os.path.abspath("..")
sys.path.append(BASE)
for d in ["data", "modeles", "resultats"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)
print("Dossier du projet :", BASE)
print("Contenu :", sorted(os.listdir(BASE)))
import json, html
import numpy as np
from kora_lib import CATEGORIES, NOMS, normaliser, expliquer

## 1. Charger le cerveau

Si le modèle KORA entraîné au notebook 2 est sur le Drive, on l'utilise. Sinon, on prend la référence simple (baseline) comme roue de secours, pour que la démo marche **toujours**.

In [ ]:
import joblib
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch, DEVICE = None, "cpu"

DOSSIER_KORA = f"{BASE}/modeles/kora-afroxlmr"
if torch is not None and os.path.exists(f"{DOSSIER_KORA}/config.json"):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained(DOSSIER_KORA)
    cerveau = AutoModelForSequenceClassification.from_pretrained(DOSSIER_KORA).to(DEVICE).eval()
    NOM_CERVEAU = "KORA AfroXLMR (affiné)"

    def probas(texte):
        enc = tokenizer([normaliser(texte)], return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
        with torch.no_grad():
            p = torch.softmax(cerveau(**enc).logits, dim=-1)[0].cpu().numpy()
        return {c: float(p[i]) for i, c in enumerate(CATEGORIES)}
else:
    baseline = joblib.load(f"{BASE}/modeles/baseline.joblib")
    NOM_CERVEAU = "baseline (TF-IDF) — entraîne KORA au notebook 2 sur GPU"

    def probas(texte):
        p = baseline.predict_proba([normaliser(texte)])[0]
        return {c: float(p[i]) for i, c in enumerate(CATEGORIES)}

print("🧠 Cerveau :", NOM_CERVEAU)
print(probas("envoie moi le code que tu vas recevoir"))

## 2. Charger les oreilles (fon et français)

- **Fon** : votre oreille entraînée au notebook 5 si elle existe, sinon `Professor/mms-300m-fongbe` (voir notebook 3)
- **Français** : `openai/whisper-small` d'OpenAI (licence MIT)

Mets `CHARGER_VOIX = False` si tu veux seulement la partie texte (plus rapide).

In [ ]:
CHARGER_VOIX = torch is not None

if CHARGER_VOIX:
    from transformers import pipeline
    import librosa
    dev = 0 if DEVICE == "cuda" else -1
    # Si vous avez entraîné votre propre oreille (notebook 5), elle est utilisée en priorité
    MODELE_FON = "Professor/mms-300m-fongbe"
    for option in ["B", "A"]:
        if os.path.exists(f"{BASE}/modeles/oreille-fon-kora-{option}/config.json"):
            MODELE_FON = f"{BASE}/modeles/oreille-fon-kora-{option}"
            break
    print("👂 Oreille fon :", MODELE_FON)
    oreille_fon = pipeline("automatic-speech-recognition", model=MODELE_FON, device=dev, chunk_length_s=20)
    oreille_fr = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=dev, chunk_length_s=30)

    def transcrire(chemin, langue):
        son, _ = librosa.load(chemin, sr=16000, mono=True)
        entree = {"raw": son, "sampling_rate": 16000}
        if langue == "fon":
            return oreille_fon(entree)["text"]
        return oreille_fr(entree, generate_kwargs={"language": "french", "task": "transcribe"})["text"]
    print("👂 Oreilles fon + français prêtes")
else:
    def transcrire(chemin, langue):
        raise RuntimeError("Les oreilles ne sont pas chargées (CHARGER_VOIX = False).")
    print("⏭ Voix désactivée : seule l'analyse de texte est disponible.")

## 3. L'affichage : la carte de verdict et la mascotte

La mascotte est la page 3D `app/kora_mascotte.html`. On lui transmet l'émotion et le texte de sa bulle.

In [ ]:
MASCOTTE = open(f"{BASE}/app/kora_mascotte.html", encoding="utf-8").read()

def mascotte(emotion="content", bulle="Colle-moi un message !"):
    reglages = (f"<script>window.KORA_EMBED=true;window.KORA_INIT={json.dumps(emotion)};"
                f"window.KORA_BULLE={json.dumps(bulle)};</script>")
    page = MASCOTTE.replace("<head>", "<head>" + reglages, 1)
    return (f'<iframe srcdoc="{html.escape(page, quote=True)}" '
            'style="width:100%;height:440px;border:0;border-radius:18px;background:#0b0b0d"></iframe>')

COULEURS = {"arnaque": "#e5484d", "suspect": "#f5a524", "sur": "#30a46c"}
BULLES = {"arnaque": "Arnaque ! Ne donne rien !", "suspect": "Hmm… méfie-toi.", "sur": "Ça a l'air normal 👍"}

def carte(r, transcription=None):
    c = COULEURS[r["niveau"]]
    raisons = "".join(f"<li>{html.escape(x)}</li>" for x in r["raisons"])
    trans = (f'<p style="opacity:.75;font-size:14px">📝 <b>Texte analysé :</b> {html.escape(transcription)}</p>'
             if transcription else "")
    return f"""
    <div style="border:3px solid {c};border-radius:18px;padding:18px 20px;font-family:system-ui,sans-serif">
      <div style="font-size:24px;font-weight:800;color:{c}">{r['titre']}</div>
      <div style="margin:10px 0 4px;font-size:14px">Risque : <b>{r['risque']:.0%}</b> · {html.escape(r['type'])}</div>
      <div style="height:10px;background:#8883;border-radius:99px;overflow:hidden">
        <div style="width:{r['risque']*100:.0f}%;height:100%;background:{c}"></div></div>
      {trans}
      {'<p style="margin:14px 0 4px;font-weight:700">Pourquoi ?</p><ul style="margin:0">' + raisons + '</ul>' if raisons else ''}
      <p style="margin-top:14px;padding:12px;border-radius:12px;background:{c}1a">💡 {html.escape(r['conseil'])}</p>
      <p style="font-size:12px;opacity:.55;margin:8px 0 0">Analyse : {html.escape(NOM_CERVEAU)}. KORA est une aide à la vigilance,
      pas une garantie absolue.</p>
    </div>"""

## 4. Les fonctions d'analyse

In [ ]:
def analyser_texte(texte):
    if not texte or not texte.strip():
        return "<p>✍️ Écris ou colle un message.</p>", mascotte("content", "Colle-moi un message !")
    r = expliquer(texte, probas(texte))
    return carte(r), mascotte(r["emotion"], BULLES[r["niveau"]])

def analyser_image(chemin):
    """Capture d'écran d'un SMS ou d'un WhatsApp -> texte (OCR) -> KORA."""
    if not chemin:
        return "", "<p>🖼️ Envoie une capture d'écran.</p>", mascotte("content", "Montre-moi la capture !")
    try:
        import pytesseract
        from PIL import Image
        texte = pytesseract.image_to_string(Image.open(chemin), lang="fra").strip()
    except Exception as e:
        return "", f"<p>❌ Lecture impossible : {html.escape(str(e))}</p>", mascotte("triste", "Je n'arrive pas à lire…")
    if not texte:
        return "", "<p>Je n'ai trouvé aucun texte dans l'image.</p>", mascotte("surpris", "Pas de texte ?")
    r = expliquer(texte, probas(texte))
    return texte, carte(r, texte), mascotte(r["emotion"], BULLES[r["niveau"]])

def analyser_audio(chemin, langue):
    if not chemin:
        return "", "<p>🎙️ Enregistre ou envoie un vocal.</p>", mascotte("content", "J'écoute…")
    try:
        texte = transcrire(chemin, "fon" if langue.startswith("Fon") else "fr")
    except Exception as e:
        return "", f"<p>❌ {html.escape(str(e))}</p>", mascotte("triste", "Je n'ai pas compris…")
    r = expliquer(texte, probas(texte))
    return texte, carte(r, texte), mascotte(r["emotion"], BULLES[r["niveau"]])

print(analyser_texte("Bjr c l'agent MTN, envoie moi le code que tu vas recevoir stp")[0][:300])

## 5. L'application 🚀

Exécute la cellule : un lien `https://….gradio.live` apparaît. Ouvre-le sur ton téléphone.
Pour arrêter : bouton ⏹ de la cellule.

In [ ]:
import gradio as gr

EXEMPLES = [
    "Bonjour, ici le service client MTN. Suite à une mise à jour, envoyez le code reçu par SMS pour sécuriser votre compte.",
    "Vous avez reçu 25 000 FCFA de DOSSOU Koffi (01 97 00 00 00). Nouveau solde : 31 500 FCFA. ID transaction : 7A3F9C21B0.",
    "Bsr, j'ai envoyé 20000 sur ton numéro par erreur c'était pour ma mère malade, renvoie moi stp 🙏",
    "Félicitations ! Votre numéro a gagné 500 000 FCFA au tirage Moov. Pour retirer, envoyez 5 000 FCFA de frais.",
    "Moov Money : votre compte a été bloqué pour activité suspecte. Cliquez sur moov-securite.top pour le réactiver.",
    "Prêt rapide sans garantie : recevez 300 000 F en 10 minutes. Payez d'abord 7 500 F de frais de dossier au 01 66 00 00 00.",
    "Maman c'est Sèna, j'ai fait le dépôt de 15 000 ce matin, regarde ton solde quand tu peux",
]

with gr.Blocks(title="KORA-lite — Bouclier anti-arnaque") as demo:
    gr.Markdown("# 🪕 KORA-lite — Bouclier anti-arnaque Mobile Money\n"
                "Colle un SMS, envoie une capture d'écran ou un vocal (fon ou français). KORA te dit si c'est une arnaque, pourquoi, et quoi faire.")
    with gr.Row():
        with gr.Column(scale=5):
            with gr.Tab("📩 Message"):
                champ = gr.Textbox(lines=4, label="Message reçu", placeholder="Colle ici le SMS ou le message WhatsApp…")
                bouton = gr.Button("Analyser avec KORA", variant="primary")
                gr.Examples(EXEMPLES, inputs=champ, label="Exemples")
            with gr.Tab("🎙️ Vocal / appel"):
                son = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Vocal")
                langue = gr.Radio(["Fon", "Français"], value="Fon", label="Langue parlée")
                bouton_son = gr.Button("Écouter et analyser", variant="primary")
                transcription = gr.Textbox(label="Ce que KORA a entendu", interactive=False)
            with gr.Tab("🖼️ Capture d'écran"):
                image = gr.Image(type="filepath", label="Capture d'écran du SMS ou du message")
                bouton_image = gr.Button("Lire et analyser", variant="primary")
                texte_lu = gr.Textbox(label="Texte lu sur la capture", interactive=False)
            resultat = gr.HTML()
        with gr.Column(scale=4):
            vue_mascotte = gr.HTML(mascotte())
    bouton.click(analyser_texte, champ, [resultat, vue_mascotte])
    champ.submit(analyser_texte, champ, [resultat, vue_mascotte])
    bouton_son.click(analyser_audio, [son, langue], [transcription, resultat, vue_mascotte])
    bouton_image.click(analyser_image, image, [texte_lu, resultat, vue_mascotte])

demo.launch(share=True, debug=False)

## ✅ Pour la présentation

1. Lance ce notebook ~15 min avant ton passage (le lien dure 72 h, mais la session Colab peut s'arrêter si elle reste inactive).
2. Ouvre le lien sur ton téléphone **et** sur l'ordinateur de projection.
3. Scénario conseillé (2 min) :
   - un **vrai SMS légitime** → KORA reste calme (pas de fausse alerte) ;
   - un **SMS d'arnaque** → KORA se fâche et explique pourquoi ;
   - un **vocal en fon** joué par l'équipe → KORA transcrit et détecte l'arnaque. 🎤 C'est le moment fort.
4. Plan B si internet tombe : des captures d'écran et une vidéo de la démo, préparées à l'avance.